# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [11]:
import pandas as pd
import numpy as np
import re

## Variable global
content = ""
sections_content = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGTh_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [12]:
section_header_pattern          = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)
s1_kv_pattern                   = re.compile(r"^([^:]*):(.*)$", re.MULTILINE)
s2_header_pattern               = re.compile(r"(Numéro)\s+(Début)\s+-\s+(Fin)\s+(Centroïde)\s+(Energie)\s+(FWHM)\s+(Surface)\s+(Incert\.)\s+(Fond sous)\s*\r?\n^\s*(du pic)\s+(\(canaux\))\s+(\(keV\))\s+(\(keV\))\s+(le pic)", re.MULTILINE)
s2_data_pattern                 = re.compile(r"^\s*([MmF]?)\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)", re.MULTILINE)
s3_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Indice\sde)\s+(Energie)\s+(Intensité)\s+(Activité)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(\W\w+\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\n", re.MULTILINE)
s3_data_pattern                 = re.compile(r"^\s*([A-Z]{1,2}-\d{1,3})?\s*(\d+\.\d+)?\s*(\d+\.\d+)(\*?)\s*(@?)\s*(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s4_header_1_pattern             = re.compile(r"^\s*(Nom du)\s+(Indice de)\s+(Activité moyenne)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(pondérée)$\n^\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s4_header_2_pattern             = re.compile(r"^\s*(Numéro)\s+(Energie)\s+(Intensité)\s+(Incert\.)\s+(Type)\s+(Nucléide)$\n^\s+(du pic)\s+(\WkeV\W)\s+(\Wcoups\Wsec\W)\s+(du pic)\s+(potentiel)$", re.MULTILINE)
s4_data_1_pattern               = re.compile(r"^\s*(X?)\s+([A-Z]{1,2}-\d{1,3})\s*(@?)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?", re.MULTILINE)
s4_data_2_pattern               = re.compile(r"^\s+([MmF])?\s*(\d+)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+\.\d+)\s*(Sum|D-Esc\.|S-Esc\.|Tol\.)?\s*([A-Z]{1,2}-\d{1,3})?\s*$", re.MULTILINE)
s5_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Energie)\s+(Intensité)\s*(LD\sEnergie)\s*(LD\snucléide)\s+(Activité)\s+(SD\sEnergie)$\n^\s+(nucléide)\s+(\WkeV\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\s+\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s5_data_pattern                 = re.compile(r"^\s*(\+?)\s*(\??)\s*(>?)\s*([A-Z]{1,2}-\d{1,3})?\s+(\d+\.\d+)(\*?)\s*(@?)\s+(\d+\.\d+)\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?|Non\sCalc)(?:\s*([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?))?\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s6_header_pattern               = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern          = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern        = re.compile(r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)", re.MULTILINE)


## Extraction des données de chaque section

### Notes
**TODO**
- Pour les sections de 2 à 6 extraire les métadonnées.
- Faire un `ffill` pour les noms de nucléide

### Setup
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [13]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [14]:
content_s1 = sections_content[sections_titles[0]]
matches = s1_kv_pattern.findall(content_s1)
data_s1 = pd.DataFrame({key.strip(): value.strip() for key, value in matches}, index=[0])

####### DATA ##########
data_s1

,Nom du fichier,Date du rapport,Nom de l’échantillon,Description,Identification,Type d'échantillon,Géométrie de mesure,Sensibilité de recherche,Zone de recherche des pics,Zone de calcul des surfaces,Tolérance sur les énergies,Quantité d’échantillon,Date de l’échantillon,Date de mesure,Temps actif,Temps réel,Temps mort,Date de l’étalonnage en énergie,Date de l’étalonnage en efficacité,Nom de la géométrie
0,C:\GENIE2K\CAMFILES\SPECTRES\STANDARDS\Standar...,02/06/2026 11:00:22,S_RGTh1_3cm,ech,,,,2.50,50 - 16384,50 - 16384,1.000 keV,2.9000E+00 g,13/02/2024 10:29:00,01/03/2024 12:08:24,253578.4 secondes,253655.5 secondes,0.03 %,02/06/2026,06/02/2026,


### S2 - RAPPORT ANALYSE DES PICS

In [15]:
def extract_header_s2(content):
    match = re.search(s2_header_pattern, content)
  
    if not match:
        return None
        
    columns = [
        "Marker",                                   # Marqueur (M, m, F)
        f"{match.group(1)} {match.group(9)}",       # Numéro du pic
        f"{match.group(2)} {match.group(10)}",      # Début (canaux)       
        f"{match.group(3)} {match.group(11)}",      # Fin (canaux)
        match.group(4),                             # Centroïde
        f"{match.group(5)} {match.group(12)}",      # Energie (keV)
        f"{match.group(6)} {match.group(13)}",      # FWHM (keV)
        match.group(7),                             # Surface
        match.group(8),                             # Incert.
        f"{match.group(8)} {match.group(13)}"       # Fond sous le pic
    ]
    
    return columns



In [16]:
def extract_data_s2(content, header):
    matches = re.findall(s2_data_pattern, content)
    df = pd.DataFrame(matches, columns=header)

    # Replace all instances of empty string with NaN in column: 'Marker'
    df['Marker'] = df['Marker'].replace('', np.nan)

    # Change column type to category for column: 'Marker'
    df = df.astype({'Marker': 'category'})

    # Change column type to float64 for columns: 'Numéro Fond sous', 'Début du pic' and 7 other columns
    df = df.astype({'Numéro Fond sous': 'float64', 'Début du pic': 'float64', 'Fin (canaux)': 'float64', 'Centroïde': 'float64', 'Energie (keV)': 'float64', 'FWHM (keV)': 'float64', 'Surface': 'float64', 'Incert.': 'float64', 'Incert. (keV)': 'float64'})

    return df


In [17]:
content_s2 = sections_content[sections_titles[1]]
header_s2 = extract_header_s2(content_s2)
data_s2 = extract_data_s2(content_s2, header_s2)

####### DATA ##########
data_s2

,Marker,Numéro Fond sous,Début du pic,Fin (canaux),Centroïde,Energie (keV),FWHM (keV),Surface,Incert.,Incert. (keV)
0,M,1.0,77.0,113.0,87.19,15.73,1.65,3440.00,217.40,18010.0
1,m,2.0,77.0,113.0,103.25,18.70,1.65,2617.00,211.69,25680.0
2,M,3.0,169.0,259.0,182.59,33.39,0.96,1464.00,179.07,12250.0
3,m,4.0,169.0,259.0,189.69,34.70,0.97,3457.00,200.62,12170.0
4,m,5.0,169.0,259.0,203.80,37.31,0.98,2311.00,179.19,12010.0
...,...,...,...,...,...,...,...,...,...,...
168,NaN,169.0,12199.0,12218.0,12208.80,2259.47,0.38,99.45,73.52,357.6
169,M,170.0,14097.0,14176.0,14124.35,2614.14,2.64,13430.00,238.15,823.7
170,m,171.0,14097.0,14176.0,14163.65,2621.41,2.64,151.10,51.43,672.8
171,NaN,172.0,14491.0,14540.0,14516.38,2686.72,3.05,1278.00,210.49,1351.0


### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

In [18]:
def extract_header_s3(content):
    matches = re.search(s3_header_pattern, content)

    if not matches:
        return None
    
    columns = [
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        "Marker *",
        "Marker @",
        f"{matches.group(4)} {matches.group(10)}",
        f"{matches.group(5)} {matches.group(11)}",
        f"{matches.group(6)} {matches.group(12)}"
    ]
   
    return columns

In [19]:
def extract_data_s3(content, header):
    matches = re.findall(s3_data_pattern, content)
    df = pd.DataFrame(matches ,columns=header)
    
    # Loaded variable 'data_s3' from kernel state

    # Replace all instances of empty string with Nan in columns: 'Nom du nucléide', 'Indice de confiance'
    df['Nom du nucléide'] = df['Nom du nucléide'].replace('', np.nan)
    df['Indice de confiance'] = df['Indice de confiance'].replace('', np.nan)

    # Change column type to float64 for columns: 'Energie (keV)', 'Intensité (%)' and 3 other columns
    df = df.astype({'Energie (keV)': 'float64', 'Intensité (%)': 'float64', 'Activité (mBq/g   )': 'float64', 'Incert. (mBq/g   )': 'float64', 'Indice de confiance': 'float64'})

    # Replace gaps forward from the previous valid value in: 'Nom du nucléide'
    df = df.fillna({'Nom du nucléide': df['Nom du nucléide'].ffill()})

    # Change column type to category for column: 'Nom du nucléide'
    df = df.astype({'Nom du nucléide': 'category'})

    # Change column type to bool for columns: 'Marker *', 'Marker @'
    df = df.astype({'Marker *': 'bool', 'Marker @': 'bool'})
    
    return df

In [20]:
content_s3 = sections_content[sections_titles[2]]
header_s3 = extract_header_s3(content_s3)
data_s3 = extract_data_s3(content_s3, header_s3)

data_s3

,Nom du nucléide,Indice de confiance,Energie (keV),Marker *,Marker @,Intensité (%),Activité (mBq/g ),Incert. (mBq/g )
0,TL-208,0.989,72.80,True,True,2.02,522.721500,44.805340
1,TL-208,NaN,74.97,True,True,3.41,6890.631000,459.577400
2,TL-208,NaN,84.90,True,True,1.51,2501.549000,207.362700
3,TL-208,NaN,277.36,True,True,6.31,169.011600,9.289074
4,TL-208,NaN,510.77,True,False,22.60,453.027200,26.499740
5,TL-208,NaN,583.19,True,False,84.50,1174.029000,53.564900
6,TL-208,NaN,763.13,True,True,1.81,263.048700,57.139640
7,TL-208,NaN,860.56,True,False,12.42,1048.383000,55.914330
8,TL-208,NaN,2614.53,True,False,99.16,1180.321000,55.557360
9,PB-210,1.000,46.53,True,False,4.25,50.461480,10.057750


### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

In [21]:
def extract_header_1_s4(content):
    matches = re.search(s4_header_1_pattern, content)
    if not matches:
        return None
    
    columns = [
        "Marker (X)",
        f"{matches.group(1)} {matches.group(5)}",
        "Marker (@)",
        f"{matches.group(2)} {matches.group(6)}",
        f"{matches.group(3)} {matches.group(7)} {matches.group(8)}",
        f"{matches.group(4)} {matches.group(9)}"
    ]
    
    return columns

In [22]:
def extract_header_2_s4(content):
    matches = re.search(s4_header_2_pattern, content)

    if not matches:
        return None

    columns = [
        "Marker (M/m/F)",
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        f"{matches.group(4)}",
        f"{matches.group(5)} {matches.group(10)}",
        f"{matches.group(6)} {matches.group(11)}",
    ]

    return columns

In [23]:
def extract_data_1_s4(content, header):
    matches = re.findall(s4_data_1_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Change column type to bool for columns: 'Marker (X)', 'Marker (@)'
    df = df.astype({'Marker (X)': 'bool', 'Marker (@)': 'bool'})

    # Replace all instances of empty string with NaN in columns: 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )'
    df['Activité moyenne pondérée (mBq/g   )'] = df['Activité moyenne pondérée (mBq/g   )'].replace('', np.nan)
    df['Incert. (mBq/g   )'] = df['Incert. (mBq/g   )'].replace('', np.nan)

    # Change column type to object for columns: 'Indice de confiance', 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )'
    df = df.astype({'Indice de confiance': 'object', 'Activité moyenne pondérée (mBq/g   )': 'object', 'Incert. (mBq/g   )': 'object'})

    # Change column type to category for column: 'Nom du nucléide'
    df = df.astype({'Nom du nucléide': 'category'})
    
    return df

In [24]:
def extract_data_2_s4(content, header):
    matches = re.findall(s4_data_2_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    

    # Replace all instances of empty string with NaN in columns: 'Marker (M/m/F)', 'Type du pic', 'Nucléide potentiel'
    df['Marker (M/m/F)'] = df['Marker (M/m/F)'].replace('', np.nan)
    df['Type du pic'] = df['Type du pic'].replace('', np.nan)
    df['Nucléide potentiel'] = df['Nucléide potentiel'].replace('', np.nan)

    # Change column type to category for columns: 'Marker (M/m/F)', 'Type du pic', 'Nucléide potentiel'
    df = df.astype({'Marker (M/m/F)': 'category', 'Type du pic': 'category', 'Nucléide potentiel': 'category'})

    # Change column type to float64 for columns: 'Numéro du pic', 'Energie (keV)' and 2 other columns
    df = df.astype({'Numéro du pic': 'float64', 'Energie (keV)': 'float64', 'Intensité (coups/sec)': 'float64', 'Incert.': 'float64'})
    
    return df

In [25]:
content_s4 = sections_content[sections_titles[3]]
header_1_s4 = extract_header_1_s4(content_s4)
header_2_s4 = extract_header_2_s4(content_s4)

data_1_s4 = extract_data_1_s4(content_s4, header_1_s4)
data_2_s4 = extract_data_2_s4(content_s4, header_2_s4)

data_2_s4

,Marker (M/m/F),Numéro du pic,Energie (keV),Intensité (coups/sec),Incert.,Type du pic,Nucléide potentiel
0,M,1.0,15.73,0.013567,6.32,NaN,NaN
1,m,2.0,18.70,0.010320,8.09,NaN,NaN
2,M,3.0,33.39,0.005772,12.23,NaN,NaN
3,m,4.0,34.70,0.013634,5.80,NaN,NaN
4,m,5.0,37.31,0.009114,7.75,NaN,NaN
...,...,...,...,...,...,...,...
122,NaN,168.0,2204.14,0.000620,61.62,NaN,NaN
123,NaN,169.0,2259.47,0.000392,73.93,NaN,NaN
124,m,171.0,2621.41,0.000596,34.02,NaN,NaN
125,NaN,172.0,2686.72,0.005041,16.47,Sum,NaN


### S5 - RAPPORT LIMITES DE DETECTION

In [26]:
def extract_header_s5(content):
    matches = re.search(s5_header_pattern, content)
    
    if not matches:
        return None
    
    columns = [
        "Marker (+)",
        "Marker (?)",
        "Marker (>)",
        f"{matches.group(1)} {matches.group(8)}",
        f"{matches.group(2)} {matches.group(9)}",
        "Marker (*)",
        "Marker (@)",
        f"{matches.group(3)} {matches.group(10)}",
        f"{matches.group(4)} {matches.group(11)}",
        f"{matches.group(5)} {matches.group(12)}",
        f"{matches.group(6)} {matches.group(13)}",
        f"{matches.group(7)} {matches.group(14)}"
    ]
    
    return columns

In [ ]:
def extract_data_s5(content, header):
    matches = re.findall(s5_data_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Loaded variable 'data_s5' from kernel state

    # Replace all instances of empty string with NaN in column: 'Nom du nucléide'
    df['Nom du nucléide'] = df['Nom du nucléide'].replace('', np.nan)
    
    # Replace all instances of 'Non Calc' with NaN in column: 'LD Energie (mBq/g   )'
    df['LD Energie (mBq/g   )'] = df['LD Energie (mBq/g   )'].replace('Non Calc', np.nan)

    # Replace all instances of empty string with NaN in column: 'LD nucléide (mBq/g   )'
    df['LD nucléide (mBq/g   )'] = df['LD nucléide (mBq/g   )'].replace('', np.nan)

    # Change column type to bool for columns: 'Marker (+)', 'Marker (?)' and 3 other columns
    df = df.astype({'Marker (+)': 'bool', 'Marker (?)': 'bool', 'Marker (>)': 'bool', 'Marker (*)': 'bool', 'Marker (@)': 'bool'})

    # Change column type to float64 for columns: 'Energie (keV)', 'Intensité (%)' and 4 other columns
    df = df.astype({'Energie (keV)': 'float64', 'Intensité (%)': 'float64', 'LD Energie (mBq/g   )': 'float64', 'LD nucléide (mBq/g   )': 'float64', 'Activité (mBq/g   )': 'float64', 'SD Energie (mBq/g   )': 'float64'})

    # Replace gaps forward from the previous valid value in: 'Nom du nucléide'
    df = df.fillna({'Nom du nucléide': df['Nom du nucléide'].ffill()})
    
    return df

In [30]:
content_s5 = sections_content[sections_titles[4]]
header_s5 = extract_header_s5(content_s5)
data_s5 = extract_data_s5(content_s5, header_s5)

data_s5

,Marker (+),Marker (?),Marker (>),Nom du nucléide,Energie (keV),Marker (*),Marker (@),Intensité (%),LD Energie (mBq/g ),LD nucléide (mBq/g ),Activité (mBq/g ),SD Energie (mBq/g )
0,False,False,False,BE-7,477.59,False,False,10.42,38.240,38.240,25.970,18.950
1,False,False,False,K-40,1460.81,False,False,10.67,24.720,24.720,1.781,12.170
2,False,False,False,CS-137,32.19,False,False,3.66,56.990,3.305,-188.100,28.010
3,False,False,False,CS-137,661.65,False,False,85.12,3.305,NaN,-15.540,1.637
4,True,False,False,TL-208,72.80,True,False,2.02,67.080,11.990,522.700,33.290
...,...,...,...,...,...,...,...,...,...,...,...,...
119,False,False,False,U-235,163.35,True,False,4.70,21.780,NaN,46.000,10.600
120,False,False,False,U-235,185.71,True,False,54.00,2.157,NaN,8.008,1.070
121,False,False,False,U-235,202.12,False,False,1.00,136.600,NaN,-343.200,67.830
122,False,False,False,U-235,205.31,False,False,4.70,30.510,NaN,-33.650,14.860


### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [ ]:
def extract_header_s6(content):
    header = []
    matches = re.findall(s6_header_pattern, content)

    l1 = re.findall(s6_word_column_pattern, matches[0][0])
    l2 = re.findall(s6_word_column_pattern, matches[0][1])
    
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header

In [ ]:
def extract_data_s6(content, header):
    matches = re.findall(s6_nucleide_line_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Change column type to category for column: 'Nucléide'
    df = df.astype({'Nucléide': 'category'})
    
    # Change column type to float64 for columns: 'LD', 'SD' and 6 other columns
    df = df.astype({'LD': 'float64', 'SD': 'float64', 'Limite Basse': 'float64', 'Limite Haute': 'float64', 'Moyenne Activité': 'float64', 'pondérée Incert.': 'float64', 'Meilleure Activité': 'float64', 'Estimation Incert.': 'float64'})

    return df

In [ ]:
content_s6 = sections_content[sections_titles[5]]
header_s6 = extract_header_s6(content_s6)
data_s6 = extract_data_s6(content_s6, header_s6)

####### DATA ##########
data_s6

# Zone de Test